# 5. Relaties, en wat maakt dat je er eentje gelooft

**De vraag: betekent dit patroon wel iets?**

Les 3 trok het saaie deel af, les 4 fitte wat er overbleef. Deze les gaat over de stap die iedereen overslaat — bepalen of wat je hebt gevonden wel echt is *vóórdat* je het opschrijft.

Drie stappen, op volgorde:

1. **De lijn die je trekt is een claim.** Een gefitte lijn vertelt in wat voor *soort* relatie je gelooft, en de grafiek laat zien of dat geloof wel verdiend was.
2. **Stap in de valkuil.** Zoek doelbewust naar een verschil tussen twee groepen die niet verschillen. Je zult er ongeveer de helft van de keren eentje vinden, en de wiskunde legt uit waarom.
3. **Een raster om te bepalen wat een bevinding waard is**, toegepast op drie claims die drie verschillende oordelen krijgen.

Samenvattende statistieken stemmen veel vaker met elkaar overeen dan grafieken — Anscombe's kwartet maakte dat punt al in 1973 en dat heb je gezien. De gewoonte die dat oplevert is wat dit notebook consequent toepast: plot het vóórdat je het samenvat, en plot het daarna nog eens.

De rest van de les ontmoet de confounder die door geen enkele methode wordt opgelost ([05.2](05.2-correlation%20jeffrey.ipynb)), modelleert een relatie die ophoudt stand te houden — overlijdens volgend op positieve tests, eerst als vraag ([05.3](05.3-covid-modelling%20jeffrey.ipynb)) en vervolgens uitgewerkt ([05.4](05.4-covid-solution%20jeffrey.ipynb)) — en loodst één claim helemaal door het raster ([05.5](05.5-how-people-type%20jeffrey.ipynb)).

In [ ]:
import numpy as np
import pandas as pd
from goad_toolkit.datatransforms import (
    Filter,
    FlagDates,
    GroupAgg,
    Pipeline,
    RegexFeature,
)
from goad_toolkit.visualizer import (
    BarbellPlot,
    BarPlot,
    CorrelationHeatmap,
    GroupedBarPlot,
    HighlightCategory,
    HistogramPlot,
    HorizontalLine,
    LinePlot,
    PlotSettings,
    RegPlot,
    ScatterPlot,
    VerticalLine,
)
from scipy import stats

from scripts.pipelines import build_irc_pipeline
from wa_analyzer.data import load_showcase

## 5.1 De lijn die je trekt is een claim

Een spreidingsdiagram toont de relatie. Een gefitte lijn zegt wat voor *soort* het is — en dat is een claim die jij maakt, geen opmaakkeuze. `RegPlot` biedt de drie opties die ertoe doen: `fit_reg` om te bepalen of je er überhaupt eentje trekt, `order` voor een polynoom, en `lowess` om de data zelf de vorm te laten bepalen.

Brandstofverbruik afgezet tegen gewicht, wat berucht géén rechte lijn is. De punten zijn met opzet grijs: de lijn is de claim, dus de lijn krijgt de kleur.

In [ ]:
mpg = load_showcase("mpg").dropna(subset=["weight", "mpg"])

shapes = PlotSettings(
    figsize=(10, 8),  # Ruime afmetingen voor een 2x2 raster
    max_cols=2,  # Dwingt maximaal 2 kolommen af -> 2x2 layout bij 4 plots
    title="Four claims about the same scatter",
    subplot_titles=[
        "order=1: a straight line",
        "order=2: quadratic curve",
        "order=3: cubic curve",
        "lowess: no shape assumed",
    ],
    xlabel="weight (lbs)",
    ylabel="miles per gallon",
)
points = {"scatter_kws": {"color": "#bbbbbb", "s": 12}}

host = ScatterPlot(shapes)
fig, axes = host.create_figure(n_plots=4)
host.plot_on_axes(RegPlot(shapes), axes[0], data=mpg, x="weight", y="mpg", ci=None, **points)
host.plot_on_axes(RegPlot(shapes), axes[1], data=mpg, x="weight", y="mpg", order=2, ci=None, **points)
host.plot_on_axes(RegPlot(shapes), axes[2], data=mpg, x="weight", y="mpg", order=3, ci=None, **points)
_ = host.plot_on_axes(RegPlot(shapes), axes[3], data=mpg, x="weight", y="mpg", lowess=True, **points)


De rechte lijn zit er op een specifieke, duidelijke manier naast: hij overschat het rendement voor de zwaarste auto's en onderschat het voor de lichtste, omdat hij een kromme probeert te middelen. De kwadratische lijn volgt de bocht. Lowess is het eens met de kwadratische lijn zonder dat hem verteld is dat er een bocht gezocht moest worden.

**Welke te gebruiken:** `lowess` eerst, wanneer je de vorm nog niet kent — dat is een beschrijving. Daarna een polynoom zodra je hebt besloten wat de vorm *is* — dat is een model, het heeft parameters, en je kunt ermee extrapoleren en er op een informatieve manier naast zitten.

De getallen achter de lijnen: `scipy.stats.linregress` voor de rechte lijn, en dezelfde fit op log-log assen, omdat "elk extra pond proportioneel minder kost" een claim is over *verhoudingen* in plaats van absolute verschillen.

In [ ]:
fit = stats.linregress(mpg.weight, mpg.mpg)
log_fit = stats.linregress(np.log(mpg.weight), np.log(mpg.mpg))
quad = np.polyfit(mpg.weight, mpg.mpg, 2)

print(f"straight line  slope {fit.slope:.5f} mpg per lb   r^2 {fit.rvalue**2:.3f}   p {fit.pvalue:.1e}")
print(f"log-log line   slope {log_fit.slope:.3f}             r^2 {log_fit.rvalue**2:.3f}")
print(f"quadratic      {quad[0]:.3e} x^2 + {quad[1]:.3f} x + {quad[2]:.1f}")

`r² = 0.69` en een p-waarde met heel veel nullen — beide waar, en geen van beide vertelt je dat de relatie gekromd is, wat het plaatje direct duidelijk maakte. De log-log fit stelt dat het verbruik (mpg) daalt met ongeveer de **macht 1,2** van het gewicht, en de r² ervan verslaat de rechte lijn op de ruwe schaal. Dat is een uitspraak over een mechanisme — een verdubbeling van het gewicht kost meer dan de helft van het rendement — in plaats van een richtingscoëfficiënt in eenheden waar niemand in denkt.

> **Jouw beurt, in het kort.** Fit `order=3` op dezelfde data. Volgt dit de punten beter? > *Voorspelt* het beter? Dat zijn twee verschillende vragen, en les 6 is waar die tweede vraag zijn eigen gereedschap krijgt.

## 5.2 De valkuil: een bevinding over gebakken lucht

**Het inzicht dat dit onderdeel duidelijk maakt:** doorzoek vijftien metrieken naar een verschil tussen twee groepen, en je zult ongeveer de helft van de keren een "significant" verschil vinden *terwijl er helemaal niets te vinden is*. Alles hieronder is die ene zin, eerst getoond en vervolgens uitgelegd.

Het IRC-corpus, één rij per `#ubuntu-uk` auteur met dertig of meer berichten, en vijftien metrieken per auteur: hoe lang hun berichten zijn, hoe vaak ze 's nachts posten, hoeveel vragen en uitroeptekens, hoeveel links, enzovoort. De patroon-gebaseerde features zijn `RegexFeature` stappen in een pipeline — de methode uit les 1 — terwijl de eenvoudige kenmerken (lengte, tijdsvlag) direct worden toegekend. Elke auteur krijgt ook een `cohort`-label: A of B.

In [ ]:
msgs = build_irc_pipeline().apply(load_showcase("ubuntu_irc"))

uk = (
    Pipeline()
    .add(Filter, expr="channel == '#ubuntu-uk'")
    .add(RegexFeature, name="words", column="message", pattern=r"\S+",
         feature="words", mode="count")
    .add(RegexFeature, name="caps", column="message", pattern=r"^[A-Z]",
         feature="starts_upper", mode="has")
    .add(RegexFeature, name="exclaims", column="message", pattern="!",
         feature="exclaims", mode="count")
    .apply(msgs)
)
uk["length"] = uk.message.str.len()
uk["night"] = uk.hh.between(0, 5)
uk["is_weekend"] = uk.date.dt.dayofweek >= 5

authors = uk.groupby("author").agg(
    n=("message", "size"),
    mean_length=("length", "mean"),
    median_length=("length", "median"),
    mean_words=("words", "mean"),
    night_share=("night", "mean"),
    upper_share=("starts_upper", "mean"),
    question_rate=("n_question", "mean"),
    exclaim_rate=("exclaims", "mean"),
    url_share=("has_url", "mean"),
    address_share=("addressed_to", lambda s: s.notna().mean()),
    active_days=("date", "nunique"),
    weekend_share=("is_weekend", "mean"),
)
authors = authors[authors.n >= 30]
authors["msgs_per_day"] = authors.n / authors.active_days
authors["unique_share"] = uk.groupby("author").message.nunique().reindex(authors.index) / authors.n
authors["hour_spread"] = uk.groupby("author").hh.std().reindex(authors.index)

METRICS = [
    "mean_length", "median_length", "mean_words", "night_share", "upper_share",
    "question_rate", "exclaim_rate", "url_share", "address_share", "active_days",
    "msgs_per_day", "unique_share", "hour_spread", "weekend_share", "n",
]

rng = np.random.default_rng(60)
authors["cohort"] = rng.permutation(["A", "B"] * (len(authors) // 2 + 1))[: len(authors)]
print(f"{len(authors)} authors, {len(METRICS)} metrics, cohorts: {authors.cohort.value_counts().to_dict()}")

Toets elke metriek tussen de cohorten — met voor elk een Welch t-toets — en zet de vijftien p-waarden op één as tegen de conventionele lijn op 0,05. De metriek onder de lijn is de "bevinding".

In [ ]:
comparison = []
for metric in METRICS:
    a = authors.loc[authors.cohort == "A", metric].dropna()
    b = authors.loc[authors.cohort == "B", metric].dropna()
    comparison.append({
        "metric": metric,
        "cohort A": a.mean(),
        "cohort B": b.mean(),
        "ratio": a.mean() / b.mean(),
        "p": stats.ttest_ind(a, b, equal_var=False).pvalue,
    })
comparison = pd.DataFrame(comparison).sort_values("p").reset_index(drop=True)
winner = comparison.metric[0]

hunt_plot = PlotSettings(
    figsize=(11, 4),
    title="Fifteen metrics, one of them below the line",
    xlabel="",
    ylabel="p-value (log scale)",
    xtick_rotation=45,
    base_color="#cccccc",
    highlight_color="crimson",
)
bars = BarPlot(hunt_plot)
fig, ax = bars.plot(data=comparison, x="metric", y="p", color=hunt_plot.base_color)
bars.plot_on(HighlightCategory(hunt_plot), categories=[winner])
bars.plot_on(HorizontalLine(hunt_plot), y=0.05, label="p = 0.05", color="black", linewidth=1)
ax.set_yscale("log")
ax.legend()

print(comparison.head(3).round(4).to_string(index=False))

Cohort A plaatst **twee keer zoveel van hun berichten tussen middernacht en 05:00** als cohort B — 13,5% van hun berichten tegenover 6,7% — met `p = 0.0026`, op meer dan tweehonderd auteurs per groep.

Dat klinkt als een publiceerbare zin, en een mechanisme dient zich onmiddellijk aan: cohort A bestaat vast uit nachtbrakers, of ze zitten in een andere tijdzone. Zie hoe snel die verklaring paraat was.

### De onthulling

Kijk eens naar de regel die de cohorten aanmaakte in de setup-cel:

    authors["cohort"] = rng.permutation(["A", "B"] * ...)

Er is helemaal geen cohort. De twee groepen zijn exact dezelfde populatie, willekeurig gesplitst, en elk verschil in die plot is door pure willekeur ontstaan.

### Waarom dit onvermijdelijk was

Een p-waarde van 0,05 betekent: *als er niets aan de hand is*, duikt een verschil van deze grootte één op de twintig keer op. Kies je die drempelwaarde, dan accepteer je een vals-alarmkans van 5% per toets. Vijftien toetsen, elk met een kans van 95% om stil te blijven, die allemaal tegelijk stil moeten blijven:

    P(geen valse bevinding in 15 toetsen) = 0.95^15 = 0.46
    P(minstens één valse bevinding)       = 1 - 0.46 = 0.54

Het complement is de truc die de moeite waard is om te onthouden: "minstens één" is lastig te tellen, "geen van alle" is een enkelvoudig product. Meet het nu in plaats van het alleen af te leiden — schud de labels driehonderd keer en tel hoeveel metrieken per zoektocht onder de 0,05 duiken.

In [ ]:
def hunt(seed: int) -> int:
    """Assign cohorts at random, then count the metrics that come out 'significant'."""
    labels = pd.Series(
        np.random.default_rng(seed).permutation(authors.cohort.to_numpy()), index=authors.index
    )
    found = 0
    for metric in METRICS:
        a = authors.loc[labels == "A", metric].dropna()
        b = authors.loc[labels == "B", metric].dropna()
        found += stats.ttest_ind(a, b, equal_var=False).pvalue < 0.05
    return found


findings = pd.Series([hunt(seed) for seed in range(300)], name="findings")
share = findings.value_counts(normalize=True).sort_index().rename("share").reset_index()

null_hunt = PlotSettings(
    figsize=(8, 4),
    title=f"300 hunts over random labels: {(findings > 0).mean():.0%} find something",
    xlabel="metrics below p = 0.05, in one hunt",
    ylabel="share of hunts",
    base_color="#cccccc",
    highlight_color="crimson",
)
bars = BarPlot(null_hunt)
fig, ax = bars.plot(data=share, x="findings", y="share", color=null_hunt.base_color)
_ = bars.plot_on(HighlightCategory(null_hunt), categories=[str(v) for v in share.findings if v > 0])

print(f"measured: a finding in {(findings > 0).mean():.0%} of hunts")
print(f"independence formula: 1 - 0.95^15 = {1 - 0.95 ** 15:.0%}")

Ongeveer twee van de vijf zoektochten leveren minstens één "bevinding" op, sommige twee of drie, en ze gaan allemaal over gebakken lucht. **Dat getal is de nuchtere baseline**, en het berekenen ervan is de gewoonte die dit onderdeel je leert: vraag vóórdat je vraagt "is mijn bevinding echt?", eerst: "hoe vaak zou ik *iets* hebben gevonden als er helemaal niets te vinden was?" Als het antwoord "bijna de helft van de keren" is, dan is het vinden van iets geen enkel bewijs van wat dan ook.

Het gemeten percentage ligt onder de 54% die de formule voorspelde, en de reden is belangrijk: de formule ging uit van vijftien *onafhankelijke* metrieken. Dat zijn ze niet — `mean_length`, `median_length` en `mean_words` zijn drie namen voor berichtgrootte; `n`, `active_days` en `msgs_per_day` zijn drie manieren om te kijken naar hoeveel iemand post. Vijftien vragen zijn geen vijftien onafhankelijke kansen als meerdere ervan dezelfde vraag stellen.

In [ ]:
heat = PlotSettings(figsize=(8, 6.5), title="The fifteen 'independent' metrics", xlabel="", ylabel="")  # ty: ignore[invalid-argument-type]
fig, ax = CorrelationHeatmap(heat).plot(data=authors[METRICS], method="spearman", annot=False)

eigenvalues = np.linalg.eigvalsh(authors[METRICS].corr().fillna(0))
effective = (eigenvalues.sum() ** 2) / (eigenvalues ** 2).sum()
print(f"effective number of independent metrics ≈ {effective:.0f}  →  1 - 0.95^{effective:.0f} = {1 - 0.95 ** effective:.0%}")

Ergens tussen de negen en elf reële kansen, geen vijftien. Beide kanten doen ertoe: uitgaan van onafhankelijkheid zorgt dat je *overschat* hoeveel je hebt gezocht, en een baseline die te hoog ligt zorgt dat een echte bevinding terzijde wordt geschoven; de baseline compleet vergeten zorgt ervoor dat je het eerste het beste opvallende ding direct gelooft.

De eerlijke versie van de bevinding: *"het grootste van vijftien verschillen die ik heb bekeken, op labels die ik niet vooraf had gekozen."* Die zin is publiceerbaar. "Cohort A zijn nachtbrakers" is dat niet.

### De verdediging: kun je de toekomst voorspellen?

Elke oplossing is een variant van hetzelfde principe — committeer je vóórdat je kijkt. Schrijf de vraag eerst op, zodat er slechts één toets is. Vermeld naar hoeveel dingen je hebt gekeken. Of, het krachtigst van alles: **splits de data**: zoek op de ene helft, bevestig op de andere. Het is de enige oplossing die een lezer kan controleren, en het verandert een claim in een voorspelling: als het patroon echt is, zit het ook in data die je nog niet hebt bekeken.

Neem de nachtbraker-bevinding mee naar de twee helften van het corpus die door de zoektocht nooit gescheiden zijn: het nachtaandeel van dezelfde auteurs in 2013–2015 en in 2016–2017.

In [ ]:
def night_share(expr: str, feature: str) -> pd.DataFrame:
    """Per-author night share, on only the rows `expr` selects."""
    return (
        Pipeline()
        .add(Filter, expr=expr)
        .add(GroupAgg, by="author", column="night", agg="mean", feature=feature)
        .apply(uk)
    )


held_out = (
    night_share("date.dt.year >= 2013", "whole period")
    .merge(night_share("date.dt.year <= 2015", "2013-2015"), on="author")
    .merge(night_share("date.dt.year >= 2016", "2016-2017"), on="author")
    .join(authors[["cohort"]], on="author")
    .dropna()
)
long = held_out.melt(id_vars=["author", "cohort"], var_name="period", value_name="night share")

for period in ["whole period", "2013-2015", "2016-2017"]:
    a = held_out.loc[held_out.cohort == "A", period]
    b = held_out.loc[held_out.cohort == "B", period]
    print(f"{period:13s} A {a.mean():.3f}  B {b.mean():.3f}   p = {stats.ttest_ind(a, b, equal_var=False).pvalue:.3f}")

predict = PlotSettings(
    figsize=(9, 4),
    title=f"The night-owl gap, on the {len(held_out)} authors active in both halves",
    xlabel="",
    ylabel="share of messages at 00-05h",
)
fig, ax = GroupedBarPlot(predict).plot(
    data=long, x="period", y="night share", hue="cohort",
    palette={"A": "crimson", "B": "#cccccc"}, errorbar=("ci", 95), capsize=0.1,
)
_ = ax.legend(title="cohort", loc="upper left", bbox_to_anchor=(1.01, 1))

Beperkt tot de auteurs die in beide helften aanwezig zijn, is het verschil nog maar een schim van het headline-getal, en in geen van beide helften overleeft het: de intervallen overlappen en de p-waarden komen nergens in de buurt van 0,05. Het was er nooit — willekeurige labels kunnen niets voorspellen over een periode die de zoektocht niet heeft geraakt.

> **De regel om te onthouden:** een bevinding en de zoektocht die haar voortbracht zijn één en hetzelfde object. > Rapporteer ze samen, anders heb je de bevinding niet gerapporteerd.

## 5.3 Wat zou maken dat je het gelooft

Voer het vorige onderdeel geïsoleerd uit en de conclusie is "niets is ooit echt", wat nog erger is dan de goedgelovigheid die het moest genezen. Dus: wat zou een bevinding wél geloofwaardig maken?

Een claim rust op drie pijlers, en statistiek levert er exact één van:

1. **Bewijs (Evidence)** — hoeveel waarnemingen, *op het niveau van de eenheid waar de claim over gaat*, hoe groot is het effect, en overleeft het een toevalstoets (nulplot)?
2. **Mechanisme (Mechanism)** — is er een reden waarom het waar zou zijn, en had je de richting al voorspeld vóórdat je keek? **Geen enkele hoeveelheid data kan deze pijler leveren.**
3. **Replicatie (Replication)** — verschijnt het ook waar je nog niet had gekeken? Een andere periode, een ander kanaal, een tweede manier om hetzelfde te meten.

Kruis de twee goedkope pijlers:

| | **doorstaat toetsing** | **faalt bij toetsing** |
| -- | -- | -- |
| **mechanisme** | Rapporteer het — en ga vervolgens pijler drie controleren. | **"Aannemelijk, onbewezen."** Zeg exact dat, en geef aan wat uitsluitsel zou geven. |
| **geen mechanisme** | **Trek geen conclusies. Ga op onderzoek uit.** Een toevalstreffer, of een confounder die je nog niet hebt benoemd — en de confounder is meestal de interessantere ontdekking. | Niets hier. Laat het vallen. |

De diagonaal ligt voor de hand. De cellen buiten de diagonaal zijn de eigenlijke les, en beide zijn dingen die studenten bijna nooit opschrijven: een nulresultaat benoemen als een nulresultaat, en een significant resultaat behandelen als een vraag in plaats van als een definitief antwoord. Drie claims, drie cellen van het raster, elk met de ene plot die de doorslag geeft.

### Claim 1 · "Release-dagen zijn drukker" — en de baseline is onderdeel van de claim

Mechanisme: sterk. De datums stonden van tevoren vast via een externe kalender waar deze dataset geen invloed op had — tien releases, verspreid over vijf jaar. De voor de hand liggende toets is elke release-dag vergelijken met een typische donderdag.

In [ ]:
RELEASES = pd.to_datetime([
    "2013-04-25", "2013-10-17", "2014-04-17", "2014-10-23", "2015-04-23",
    "2015-10-22", "2016-04-21", "2016-10-13", "2017-04-13", "2017-10-19",
])

thursdays = (
    Pipeline()
    .add(Filter, expr="day_name == 'Thursday'")
    .add(FlagDates, column="date", dates=RELEASES, feature="is_release")
    .apply(uk)
)
thursday_counts = Pipeline().add(GroupAgg, by=["date", "is_release"]).apply(thursdays)

global_median = thursday_counts.loc[~thursday_counts.is_release, "n"].median()
release_counts = thursday_counts.loc[thursday_counts.is_release, "n"]
print(f"median ordinary Thursday, whole corpus: {global_median:.0f} messages")
print(f"release days above it: {(release_counts > global_median).sum()}/{len(release_counts)}")

Zes van de tien. Een muntworp, en op zichzelf schiet het de claim af.

Dat zou niet moeten, en de volgende claim verklaart waarom: dit kanaal verloor het leeuwendeel van zijn verkeer over de hele periode, waardoor "een typische donderdag" in 2013 en in 2017 totaal verschillende grootheden zijn, en de releases van 2017 worden vergeleken met een mediaan die grotendeels uit 2013 stamt. **De langetermijntrend is een confounder voor het event.** De oplossing is om elke release te vergelijken met de donderdagen er direct omheen.

In [ ]:
def local_baseline(frame: pd.DataFrame, weeks: int = 4) -> pd.DataFrame:
    """Each release against the median of the Thursdays within `weeks` either side."""
    counts = frame[frame.date.dt.dayofweek == 3].groupby("date").size().sort_index()
    rows = []
    for release_date in RELEASES:
        if release_date not in counts.index:
            continue
        window = counts[(counts.index >= release_date - pd.Timedelta(weeks=weeks))
                        & (counts.index <= release_date + pd.Timedelta(weeks=weeks))]
        rows.append({"release": str(release_date.date()),
                     "nearby Thursday": window.drop(release_date).median(),
                     "on the day": counts[release_date]})
    out = pd.DataFrame(rows)
    out["lift"] = out["on the day"] / out["nearby Thursday"]
    return out


for channel in sorted(msgs.channel.unique()):
    table = local_baseline(msgs[msgs.channel == channel])
    above = int((table.lift > 1).sum())
    p = stats.binomtest(above, len(table), 0.5).pvalue
    print(f"{channel}: {above}/{len(table)} releases above their local baseline, "
          f"median lift {table.lift.median():.2f}x, sign test p = {p:.3f}")

local = local_baseline(uk)
gap = PlotSettings(
    figsize=(9, 5),
    title="#ubuntu-uk: each release against the Thursdays around it",
    xlabel="messages on the day",
    ylabel="",
)
fig, ax = BarbellPlot(gap).plot(
    data=local.sort_values("release", ascending=False), category="release",
    start="nearby Thursday", end="on the day",
    start_label="nearby Thursdays (median)", end_label="release day",
)

**Negen van de tien**, mediane stijging 1,7×, `p = 0.02` volgens een tekentoets (*sign test*) — het juiste instrument, omdat het enkel vraagt *aan welke kant van het typische* elke dag valt, wat een vraag is die tien ruizige dagen wél kunnen beantwoorden. Dezelfde data, dezelfde claim; het enige dat veranderde was wat "typisch" betekent.

**Oordeel: rapporteren** — het dichtste dat dit notebook komt bij een bevinding die je zou kunnen publiceren. Mechanisme sterk, datums door iemand anders vastgesteld, tien nagenoeg onafhankelijke instanties. Pijler drie is gedeeltelijk aanwezig, en zeg dat ook eerlijk: `#ubuntu-nl` beweegt op dezelfde manier, maar is een tiende van de omvang, en een tiende van het verkeer levert een tiende van de resolutie op. "De richting komt overeen op een tweede kanaal dat te klein is om statistisch te toetsen" is de accurate zin.

En de omweg is de échte les. De eerste versie zei *zes van de tien* en zou zijn gerapporteerd als "geen effect". Een bevinding kan net zo makkelijk worden vernietigd door een baseline als worden verzonnen door een baseline, en daarom heeft de regel uit §5.2 zijn spiegelbeeld nodig: **de vergelijking die je kiest is onderdeel van de claim.**

### Claim 2 · "Het kanaal werd stiller na 2016"

Mechanisme: sterk, en van tevoren voorspelbaar — Slack, Discord en Matrix overkwamen IRC overal, niet alleen hier. De richting was voorspelbaar vóórdat we keken. Twee kanalen, het volume van elk jaar relatief ten opzichte van het eigen jaar 2013, op één as.

In [ ]:
per_year = (
    Pipeline()
    .add(GroupAgg, by=["channel", "year"])
    .apply(msgs.assign(year=msgs.date.dt.year))
)
per_year["relative to 2013"] = per_year.n / per_year.groupby("channel").n.transform("first")

decline = PlotSettings(
    figsize=(9, 4),
    title="Both channels, every year lower than the last",
    xlabel="",
    ylabel="messages, relative to 2013",
)
lines = LinePlot(decline)
fig, ax = lines.plot(data=per_year, x="year", y="relative to 2013", hue="channel", marker="o",
                     palette={"#ubuntu-uk": "steelblue", "#ubuntu-nl": "darkorange"})
lines.plot_on(HorizontalLine(decline), y=1, color="grey", linewidth=0.8)
ax.set_xticks(sorted(per_year.year.unique()))
ax.legend(title="")
print(per_year.pivot(index="year", columns="channel", values="relative to 2013").round(2).to_string())

`#ubuntu-uk` eindigt op **16%** van zijn volume in 2013, `#ubuntu-nl` op **8%**, en beide dalen in elk afzonderlijk jaar na 2014. Twee kanalen, verschillende landen, verschillende groottes, dezelfde richting, geen enkele uitzondering.

**Oordeel: alle drie de pijlers.** Mechanisme voorspeld vóór het kijken, effect gigantisch en monotoon, gerepliceerd op een onafhankelijk kanaal. Niets hier heeft een p-waarde nodig, en erom vragen zou een categoriefout zijn — je bent dit niet aan het onderscheiden van ruis, je kijkt er rechtstreeks naar. De eerlijke nuance hoort in dezelfde alinea thuis: dit is *dit corpus* dat stiller wordt, wat niet automatisch betekent dat IRC als geheel stiller wordt, en al helemaal niet dat de Ubuntu-gemeenschap krimpt. Mensen zijn verhuisd naar andere platforms. De claim gaat over een kanaal.

### Claim 3 · "Mensen schrijven langere berichten op doordeweekse dagen"

Mechanisme: reëel, maar **dubbelzinnig van teken (richting)**. Doordeweeks chatten gebeurt op het werk, in korte bursts tussen andere taken door — dat voorspelt kortere berichten. Doordeweeks chatten gaat ook óver werk, technisch en gedetailleerd — dat voorspelt langere berichten. Je kunt het op voorhand oprecht niet voorspellen, en dán bewijst een statistische toets zijn waarde.

De valkuil is de eenheid van analyse. Een half miljoen berichten levert een p-waarde op met negen nullen; maar de claim gaat over *mensen*, en 425.000 berichten zijn geen 425.000 onafhankelijke waarnemingen van mensen. Stel de vraag op het niveau van de eenheid waar de claim over gaat: één getal per auteur, weekdag min weekend.

In [ ]:
weekday = uk.loc[~uk.is_weekend, "length"]
weekend = uk.loc[uk.is_weekend, "length"]
print(f"per MESSAGE: weekday {weekday.mean():.2f} chars, weekend {weekend.mean():.2f}, "
      f"p = {stats.ttest_ind(weekday, weekend, equal_var=False).pvalue:.0e}  (n = {len(uk):,})")

by_author = uk.groupby(["author", "is_weekend"]).length.mean().unstack().dropna()
by_author = by_author[uk.groupby("author").size().reindex(by_author.index) >= 30]
by_author.columns = pd.Index(["weekday", "weekend"])
by_author["difference"] = by_author.weekday - by_author.weekend

paired = stats.ttest_rel(by_author.weekday, by_author.weekend)
longer = (by_author.difference > 0).mean()
print(f"per AUTHOR:  mean difference {by_author.difference.mean():+.2f} chars, "
      f"paired p = {paired.pvalue:.3f}, longer on weekdays: {longer:.0%}  (n = {len(by_author)})")

split = PlotSettings(
    figsize=(9, 4),
    title=f"One number per author: {longer:.0%} go one way, {1 - longer:.0%} the other",
    xlabel="mean message length, weekday − weekend (characters)",
    ylabel="authors",
)
hist = HistogramPlot(split)
fig, ax = hist.plot(data=by_author.difference.to_numpy(), bins=40, stat="count", color="#cccccc")
_ = hist.plot_on(VerticalLine(split), x=0, color="black", linewidth=1)

259 mensen, `p = 0.06`, en het histogram is de bevinding: een zuivere muntworp aan weerszijden van nul. Wat er ook gaande is, het is niet iets wat individuele mensen doen.

**Oordeel: aannemelijk, onbewezen.** Schrijf die zin op in een rapport, als het resultaat:

> *"De berichtlengte verschilt niet betekenisvol tussen doordeweekse dagen en weekenden op auteursniveau (n = 259, p = 0.06, en de richting splitst 53/47). Het verschil per bericht is statistisch significant, maar weerspiegelt het aantal berichten, niet de mensen. Om dit te ontwarren zijn meer auteurs nodig, of een binnen-persoon onderzoeksopzet over meerdere weken."*

Dat is een afgerond stuk werk. Studenten leveren dit bijna nooit in, omdat een nulresultaat aanvoelt als een mislukte opdracht — dus blijven ze snijden en filteren tot er iets opduikt, wat exact de jacht uit §5.2 is. **Een nulresultaat, eerlijk afgebakend, is een volwaardige voldoende.**

### Nog drie claims, elk in één rij

Het raster heeft ook antwoorden voor claims waar dit notebook geen grafiek voor nodig heeft om ze te beslechten:

| claim | mechanisme | bewijs | oordeel |
| -- | -- | -- | -- |
| "Er zijn veel minder berichten om 04:00 's nachts" | overweldigend | 15× | **waar, niet het rapporteren waard** — niemands overtuiging verschuift; wat had je *verwacht*, en waarin wijkt de data daarvan af? |
| "Nicknames die beginnen met A–M schrijven langere berichten" | geen | niets, in tien kanaaljaren — en de acht die "overeenstemmen" zijn dezelfde auteurs tien keer geteld | **laat het vallen** |
| "Mensen op de 4e verdieping hebben een slechter sentiment" (een echte inlevering) | geen *voor de verdieping* | elke reeks getallen heeft een minimum | **ga op onderzoek uit** — de verdieping is een proxy voor een afdeling, een manager, a deadline; de confounder is de echte bevinding |

Drie claims met een plot en drie zonder, en het antwoord was elke keer anders. "Wees sceptisch" had er slechts eentje goed gehad.

> **Wat je moet meenemen.** Zeg hardop vóórdat je een bevinding opschrijft: wat is het mechanisme, wat zou me hebben verrast, en waar kan ik dit controleren waar ik nog niet heb gekeken? Als het antwoord op de eerste "geen" is, ben je niet klaar — je staat aan het begin van een veel interessantere vraag. En één vraag ligt onder dit alles, dezelfde vraag die les 4 beantwoordde door te shufflen: **wat zou ik verwachten te zien als hier helemaal niets aan de hand was?**